---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

## Konfiguracja środowiska i sesji Spark <a id='krok-0'></a>

In [33]:
import os
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import (
    col, to_timestamp, hour, count, desc,
    broadcast, when, udf,
)
from pyspark.sql.window import Window


SRC_CSV = "/content/chicago_crimes_sample.csv"
OUT_DIR = "output_parquet"

In [34]:
!pwd

/content


In [35]:
spark = (
    SparkSession.builder
    .appName("ChicagoCrimesAnalysis")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"PySpark wersja: {spark.version}")

PySpark wersja: 4.0.2


## Wczytanie i czyszczenie danych <a id='krok-1'></a>

Kolejność operacji czyszczenia:
1. Wczytanie CSV z `inferSchema`
2. Ujednolicenie nazw kolumn (małe litery)
3. `dropDuplicates(["id"])` – usunięcie duplikatów po kluczu głównym
4. `dropna(subset=KEY_COLS)` – usunięcie wierszy z brakami w kluczowych kolumnach
5. Parsowanie daty i filtrowanie błędnych wierszy
6. Filtrowanie zakresu lat (2000–2030) – ochrona przed błędami ETL
7. **`cache()`** – DataFrame będzie wielokrotnie odpytywany, opłaca się go zatrzymać w RAM

In [36]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SRC_CSV)
)

# Ujednolicenie nazw kolumn
df_raw = df_raw.toDF(*[c.lower().replace(" ", "_") for c in df_raw.columns])

raw_count = df_raw.count()
print(f"Wiersze wczytane (surowe):   {raw_count:>8,}")
print(f"Liczba kolumn:               {len(df_raw.columns):>8}")
print()
df_raw.printSchema()

Wiersze wczytane (surowe):    149,848
Liczba kolumn:                     22

root
 |-- id: string (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: integer (nullable = true)
 |-- y_coordinate: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- updated_on: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)



In [37]:
# Usuwanie duplikatów po kluczu głównym 'id'
df_no_dup = df_raw.dropDuplicates(["id"])
print(f"Usunięto duplikatów (id):    {raw_count - df_no_dup.count():>8,}")

Usunięto duplikatów (id):      65,729


In [38]:
# Usuwanie wierszy z brakami w kluczowych kolumnach
KEY_COLS = ["id", "date", "primary_type", "location_description", "year"]
df_clean = df_no_dup.dropna(subset=KEY_COLS)
print(f"Usunięto wierszy z brakami:  {df_no_dup.count() - df_clean.count():>8,}")

Usunięto wierszy z brakami:    34,316


In [39]:
df_clean = df_clean.withColumn(
    "date_ts",
    to_timestamp(col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS")
)
df_valid = df_clean.filter(col("date_ts").isNotNull())
print(f"Usunięto błędnych dat:       {df_clean.count() - df_valid.count():>8,}")

Usunięto błędnych dat:              0


In [40]:
df_valid = df_valid.filter((col("year") >= 2000) & (col("year") <= 2030))
df_valid = df_valid.withColumn("hour_of_day", hour(col("date_ts")))

# ─── CACHE ───────────────────────────────────────────────────────────────
# df_valid będzie wielokrotnie używany w krokach 2–5.
# Bez cache Spark za każdym razem czyta CSV od nowa i powtarza transformacje.
# Cache zapisuje dane w RAM (lub na dysku gdy brak RAM).
df_valid.cache()

final_count = df_valid.count()
print(f"Wiersze po czyszczeniu:    {final_count:>8,}  ← główny DataFrame")
print(f"   DataFrame zcachowany: df_valid.cache()")

Wiersze po czyszczeniu:      49,803  ← główny DataFrame
   DataFrame zcachowany: df_valid.cache()


In [41]:
df_valid.select(
    "id", "date_ts", "hour_of_day", "primary_type",
    "location_description", "arrest", "year"
).show(5, truncate=False)

+--------+-------------------+-----------+-------------+--------------------+------+----+
|id      |date_ts            |hour_of_day|primary_type |location_description|arrest|year|
+--------+-------------------+-----------+-------------+--------------------+------+----+
|14110378|2026-02-14 08:10:00|8          |OTHER OFFENSE|STREET              |false |2026|
|14110381|2026-02-14 05:00:00|5          |BATTERY      |RESIDENCE           |false |2026|
|14110410|2026-02-14 06:52:00|6          |BATTERY      |APARTMENT           |true  |2026|
|14110419|2026-02-14 09:00:00|9          |OTHER OFFENSE|APARTMENT           |false |2026|
|14110463|2026-02-14 09:09:00|9          |ASSAULT      |STREET              |true  |2026|
+--------+-------------------+-----------+-------------+--------------------+------+----+
only showing top 5 rows


## Klasyfikacja pory dnia <a id='krok-2'></a>

| Godzina | Pora dnia |
|---------|-----------|
| 6 – 12  | ranek     |
| 12 – 18 | południe  |
| 18 – 22 | wieczór   |
| 22 – 6  | noc       |

In [42]:
@udf(StringType())
def classify_time_of_day(h: int) -> str:
    """Klasyfikuje godzinę na cztery pory dnia."""
    if h is None:
        return "nieznana"
    if 6 <= h < 12:
        return "ranek (6–12)"
    elif 12 <= h < 18:
        return "południe (12–18)"
    elif 18 <= h < 22:
        return "wieczór (18–22)"
    else:
        return "noc (22–6)"

df_valid = df_valid.withColumn("time_of_day", classify_time_of_day(col("hour_of_day")))

print("Przykładowe wiersze z nową kolumną 'time_of_day':")
df_valid.select("date", "hour_of_day", "time_of_day", "primary_type").show(8, truncate=False)

Przykładowe wiersze z nową kolumną 'time_of_day':
+-------------------+-----------+------------+-----------------+
|date               |hour_of_day|time_of_day |primary_type     |
+-------------------+-----------+------------+-----------------+
|2026-02-14 08:10:00|8          |ranek (6–12)|OTHER OFFENSE    |
|2026-02-14 05:00:00|5          |noc (22–6)  |BATTERY          |
|2026-02-14 06:52:00|6          |ranek (6–12)|BATTERY          |
|2026-02-14 09:00:00|9          |ranek (6–12)|OTHER OFFENSE    |
|2026-02-14 09:09:00|9          |ranek (6–12)|ASSAULT          |
|2026-02-14 09:47:00|9          |ranek (6–12)|WEAPONS VIOLATION|
|2026-02-14 06:00:00|6          |ranek (6–12)|BURGLARY         |
|2026-02-14 07:10:00|7          |ranek (6–12)|BURGLARY         |
+-------------------+-----------+------------+-----------------+
only showing top 8 rows


## Broadcast join + zapis do Parquet <a id='krok-3'></a>

### Broadcast join
Mała tabela słownikowa (kody FBI → polskie opisy) jest **wysyłana do każdego executora**.  
Spark nie wykonuje kosztownego shuffla – optymalne gdy mniejsza tabela ma < kilkaset MB.

### Zapis do Parquet z partycjonowaniem po roku
Zalety `partitionBy("year")`:
- **Partition pruning** – zapytania filtrujące po roku pomijają niepotrzebne pliki
- Równoległy zapis i odczyt partycji
- Efektywna kompresja kolumnowa (Parquet)

In [43]:
fbi_dict = spark.createDataFrame([
    ("01A", "Morderstwo"),          ("02",  "Gwałt"),
    ("03",  "Napad z bronią"),      ("04A", "Napaść kwalifikowana"),
    ("04B", "Napaść prosta"),       ("05",  "Włamanie"),
    ("06",  "Kradzież"),            ("07",  "Kradzież pojazdu"),
    ("08A", "Podpalenie kwal."),    ("08B", "Podpalenie inne"),
    ("09",  "Inne przestępstwo"),   ("10",  "Prostytucja"),
    ("11",  "Przestępstwo sex."),   ("12",  "Narkotyki"),
    ("14",  "Naruszenie porządku"), ("15",  "Fałszerstwo"),
    ("16",  "Inne"),                ("17",  "Oszustwo"),
    ("18",  "Różne"),               ("19",  "Inne"),
    ("20",  "Inne"),                ("22",  "Inne"),
    ("24",  "Inne"),                ("26",  "Inne"),
], ["fbi_code", "fbi_kategoria_pl"])

df_enriched = df_valid.join(broadcast(fbi_dict), on="fbi_code", how="left")


df_enriched.cache()
print("Broadcast join z słownikiem FBI wykonany.")

Broadcast join z słownikiem FBI wykonany.


In [44]:
(
    df_enriched
    .write
    .mode("overwrite")
    .partitionBy("year")
    .parquet(OUT_DIR)
)

parts = sorted([p for p in os.listdir(OUT_DIR) if p.startswith("year=")])
print(f"Parquet zapisany w: {OUT_DIR}/")
print(f"Partycje: {parts}")

Parquet zapisany w: output_parquet/
Partycje: ['year=2026']


In [45]:
df_from_parquet = spark.read.parquet(OUT_DIR)
print(f"Weryfikacja – wierszy w Parquet: {df_from_parquet.count():,}")
df_from_parquet.printSchema()

Weryfikacja – wierszy w Parquet: 49,803
root
 |-- fbi_code: string (nullable = true)
 |-- id: string (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- x_coordinate: integer (nullable = true)
 |-- y_coordinate: integer (nullable = true)
 |-- updated_on: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)
 |-- date_ts: timestamp (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-

## Top 10 typów przestępstw <a id='krok-4a'></a>

In [46]:
top_crimes = (
    df_enriched
    .groupBy("primary_type")
    .agg(count("*").alias("liczba"))
    .orderBy(desc("liczba"))
    .limit(10)
)
top_crimes.show(truncate=False)

+-------------------+------+
|primary_type       |liczba|
+-------------------+------+
|THEFT              |10920 |
|BATTERY            |9253  |
|CRIMINAL DAMAGE    |5497  |
|ASSAULT            |4564  |
|MOTOR VEHICLE THEFT|3886  |
|OTHER OFFENSE      |3459  |
|BURGLARY           |3163  |
|DECEPTIVE PRACTICE |2486  |
|NARCOTICS          |1455  |
|CRIMINAL TRESPASS  |1193  |
+-------------------+------+



## Przestępstwa wg pory dnia × lokacji <a id='krok-4b'></a>

Dla każdej pory dnia wyznaczamy **top 5 lokacji** z największą liczbą zdarzeń.  
Używamy **Window function** `rank()` z partycjonowaniem po `time_of_day`.

In [47]:
w_pora = Window.partitionBy("time_of_day").orderBy(desc("liczba"))

crimes_time_loc = (
    df_enriched
    .groupBy("time_of_day", "location_description")
    .agg(count("*").alias("liczba"))
    .withColumn("rank", F.rank().over(w_pora))
    .filter(col("rank") <= 5)
    .drop("rank")
    .orderBy("time_of_day", desc("liczba"))
)
crimes_time_loc.show(25, truncate=False)

+----------------+--------------------------------------+------+
|time_of_day     |location_description                  |liczba|
+----------------+--------------------------------------+------+
|noc (22–6)      |STREET                                |4584  |
|noc (22–6)      |APARTMENT                             |2749  |
|noc (22–6)      |RESIDENCE                             |1493  |
|noc (22–6)      |PARKING LOT / GARAGE (NON RESIDENTIAL)|455   |
|noc (22–6)      |SIDEWALK                              |440   |
|południe (12–18)|STREET                                |3528  |
|południe (12–18)|APARTMENT                             |2927  |
|południe (12–18)|RESIDENCE                             |1779  |
|południe (12–18)|SIDEWALK                              |937   |
|południe (12–18)|SMALL RETAIL STORE                    |784   |
|ranek (6–12)    |STREET                                |2521  |
|ranek (6–12)    |APARTMENT                             |2329  |
|ranek (6–12)    |RESIDEN

## Aresztowania wg roku <a id='krok-4c'></a>

In [48]:
arrest_by_year = (
    df_enriched
    .groupBy("year", "arrest")
    .agg(count("*").alias("liczba"))
    .orderBy("year", "arrest")
)
arrest_by_year.show()

+----+------+------+
|year|arrest|liczba|
+----+------+------+
|2026| false| 42398|
|2026|  true|  7405|
+----+------+------+



## Ciężka agregacja + `explain()` <a id='krok-4d'></a>

Wielowymiarowa agregacja: `primary_type × location_description × time_of_day × year`  
z liczbą przestępstw, aresztowań i procentem aresztowań.

### Co widać w planie zapytania (`explain`)?

| Element planu | Co oznacza |
|---|---|
| `InMemoryTableScan` | Spark czyta z cache, **nie** wczytuje CSV od nowa |
| `BroadcastHashJoin` | Join ze słownikiem FBI bez shuffla |
| `HashAggregate` (x2) | Dwuetapowa agregacja: *partial* na każdym executorze → *final* po shuffle |
| `Exchange` | Shuffle danych między executorami (konieczny przed finalną agregacją) |
| `BatchEvalPython` | Wywołanie UDF (Python) – poza JVM codegen |

In [49]:
heavy_query = (
    df_enriched
    .groupBy("primary_type", "location_description", "time_of_day", "year")
    .agg(
        count("*").alias("liczba"),
        F.sum(when(col("arrest") == True, 1).otherwise(0)).alias("aresztowania"),
        F.round(
            F.sum(when(col("arrest") == True, 1).otherwise(0)) /
            count("*") * 100, 2
        ).alias("pct_aresztowan")
    )
    .filter(col("liczba") >= 5)
    .orderBy(desc("liczba"))
)


print("=" * 60)
print("PLAN FIZYCZNY ZAPYTANIA")
print("=" * 60)
heavy_query.explain(mode="simple")

PLAN FIZYCZNY ZAPYTANIA
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [liczba#28551L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(liczba#28551L DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=6044]
      +- Filter (liczba#28551L >= 5)
         +- HashAggregate(keys=[primary_type#23081, location_description#23083, time_of_day#24761, year#23093], functions=[count(1), sum(CASE WHEN arrest#23084 THEN 1 ELSE 0 END)])
            +- Exchange hashpartitioning(primary_type#23081, location_description#23083, time_of_day#24761, year#23093, 8), ENSURE_REQUIREMENTS, [plan_id=6040]
               +- HashAggregate(keys=[primary_type#23081, location_description#23083, time_of_day#24761, year#23093], functions=[partial_count(1), partial_sum(CASE WHEN arrest#23084 THEN 1 ELSE 0 END)])
                  +- InMemoryTableScan [primary_type#23081, location_description#23083, arrest#23084, year#23093, time_of_day#24761]
                        +- InMemoryRelation [fbi_code

In [50]:
# Wyniki ciężkiej agregacji
print("Top 20 kombinacji wg liczby przestępstw:")
heavy_query.show(20, truncate=False)

Top 20 kombinacji wg liczby przestępstw:
+-------------------+--------------------+----------------+----+------+------------+--------------+
|primary_type       |location_description|time_of_day     |year|liczba|aresztowania|pct_aresztowan|
+-------------------+--------------------+----------------+----+------+------------+--------------+
|MOTOR VEHICLE THEFT|STREET              |noc (22–6)      |2026|1068  |36          |3.37          |
|BATTERY            |APARTMENT           |noc (22–6)      |2026|1030  |254         |24.66         |
|CRIMINAL DAMAGE    |STREET              |noc (22–6)      |2026|873   |37          |4.24          |
|MOTOR VEHICLE THEFT|STREET              |wieczór (18–22) |2026|771   |17          |2.2           |
|BATTERY            |APARTMENT           |południe (12–18)|2026|744   |138         |18.55         |
|THEFT              |STREET              |noc (22–6)      |2026|728   |9           |1.24          |
|MOTOR VEHICLE THEFT|STREET              |południe (12–18)|

## Random Forest (wieloklasowy) <a id='krok-5'></a>

Cel: predykcja **typu przestępstwa** (`primary_type`) na podstawie:
- lokalizacji (`location_description`)
- godziny dnia (`hour_of_day`)
- czy nastąpiło aresztowanie (`arrest`)
- czy to zdarzenie domowe (`domestic`)
- rejonu patrolowego (`beat`, `district`, `community_area`)

### Pipeline MLlib
```
StringIndexer(primary_type → label)
StringIndexer(location_description → loc_idx)
VectorAssembler([loc_idx, hour_of_day, arrest_int, ...] → features)
RandomForestClassifier(50 drzew, maxDepth=8)
```

In [51]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Top 8 najczęstszych klas (dla lepszej jakości modelu)
top8 = [
    r["primary_type"] for r in
    df_enriched
    .groupBy("primary_type").count()
    .orderBy(desc("count")).limit(8).collect()
]
print(f"Klasy do predykcji ({len(top8)}):")
for i, c in enumerate(top8, 1):
    print(f"  {i}. {c}")

Klasy do predykcji (8):
  1. THEFT
  2. BATTERY
  3. CRIMINAL DAMAGE
  4. ASSAULT
  5. MOTOR VEHICLE THEFT
  6. OTHER OFFENSE
  7. BURGLARY
  8. DECEPTIVE PRACTICE


In [52]:
df_ml = (
    df_enriched
    .filter(col("primary_type").isin(top8))
    .select(
        "primary_type", "location_description",
        "hour_of_day", "arrest", "domestic",
        "beat", "district", "community_area"
    )
    .dropna()
    .withColumn("arrest_int",   col("arrest").cast("int"))
    .withColumn("domestic_int", col("domestic").cast("int"))
)

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Zbiór treningowy: {train_df.count():,} wierszy")
print(f"Zbiór testowy:    {test_df.count():,} wierszy")

Zbiór treningowy: 34,751 wierszy
Zbiór testowy:    8,477 wierszy


In [53]:
n_locations = df_ml.select("location_description").distinct().count()
max_bins = max(128, n_locations + 10)

type_idx = StringIndexer(inputCol="primary_type",         outputCol="label",   handleInvalid="keep")
loc_idx  = StringIndexer(inputCol="location_description", outputCol="loc_idx", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["loc_idx", "hour_of_day", "arrest_int", "domestic_int",
               "beat", "district", "community_area"],
    outputCol="features"
)

rf = RandomForestClassifier(
    labelCol="label", featuresCol="features",
    numTrees=50, maxDepth=8, maxBins=max_bins, seed=42
)

pipeline = Pipeline(stages=[type_idx, loc_idx, assembler, rf])

print("Trenowanie modelu… (ok. 1–2 min)")
model = pipeline.fit(train_df)
print("Model wytrenowany!")

Trenowanie modelu… (ok. 1–2 min)
Model wytrenowany!


In [54]:
predictions = model.transform(test_df)

acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
).evaluate(predictions)

print(f"\n{'='*40}")
print(f"  Accuracy:  {acc:.4f}  ({acc*100:.1f}%)")
print(f"  F1-Score:  {f1:.4f}")
print(f"{'='*40}")


  Accuracy:  0.4134  (41.3%)
  F1-Score:  0.3397


In [55]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

metrics = {
    "Accuracy":               evaluator.setMetricName("accuracy").evaluate(predictions),
    "F1 (ważona)":            evaluator.setMetricName("f1").evaluate(predictions),
    "Precision (ważona)":     evaluator.setMetricName("weightedPrecision").evaluate(predictions),
    "Recall (ważony)":        evaluator.setMetricName("weightedRecall").evaluate(predictions),
    "Log-Loss":               evaluator.setMetricName("logLoss").evaluate(predictions),
    "Hamming Loss":           evaluator.setMetricName("hammingLoss").evaluate(predictions),
}

print(f"\n{'Metryka':<28} {'Wartość':>8}")
print("-" * 38)
for name, val in metrics.items():
    print(f"  {name:<26} {val:>8.4f}")
print("-" * 38)


Metryka                       Wartość
--------------------------------------
  Accuracy                     0.4134
  F1 (ważona)                  0.3397
  Precision (ważona)           0.4138
  Recall (ważony)              0.4134
  Log-Loss                     1.6001
  Hamming Loss                 0.5866
--------------------------------------


In [56]:
rf_model = model.stages[-1]
feat_names = ["loc_idx", "hour_of_day", "arrest_int", "domestic_int",
              "beat", "district", "community_area"]

print("Ważność cech:")
for name, imp in sorted(zip(feat_names, rf_model.featureImportances.toArray()),
                        key=lambda x: -x[1]):
    bar = "█" * int(imp * 50)
    print(f"  {name:<20} {imp:.4f}  {bar}")

Ważność cech:
  loc_idx              0.4266  █████████████████████
  domestic_int         0.3462  █████████████████
  hour_of_day          0.0644  ███
  community_area       0.0572  ██
  arrest_int           0.0465  ██
  beat                 0.0411  ██
  district             0.0180  


In [58]:
spark.stop()
print("Spark zatrzymany.")

Spark zatrzymany.
